In [0]:
from sodapy import Socrata
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import json
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DoubleType

APP_TOKEN = dbutils.secrets.get("biz-survival", "nyc_app_token")
lake_table_path = "data_515.default.nyc_311_service_requests"

try:
    df_existing = spark.table("data_515.default.`311_service_requests`")
    last_created_date_str = df_existing.agg(F.max('created_date')).collect()[0][0]
    last_created_date = last_created_date_str.strftime("%Y-%m-%dT00:00:00.000")
    columns_str = ','.join(df_existing.columns) if isinstance(df_existing.columns, list) else df_existing.columns
except:
    # Default to the start of the current month
    last_created_date = datetime.today().replace(day=1).strftime("%Y-%m-%dT00:00:00.000")
    column_set = "*"

print("Last Created Date: ", last_created_date)

client = Socrata("data.cityofnewyork.us", APP_TOKEN)
soql_query = f"SELECT {columns_str} WHERE created_date > '{last_created_date}' limit 500000"
results = client.get("erm2-nwe9", query=soql_query)

if results:
    sample_record = results[0] if results else {}
    nyc_311_schema = StructType([StructField(k, StringType(), True) for k in sample_record.keys()])
    df_new_data = spark.createDataFrame(results, schema=nyc_311_schema)
    df_new_data = df_new_data.withColumn("created_date", F.to_timestamp("created_date"))\
                             .withColumn("closed_date", F.to_timestamp("closed_date"))\
                             .withColumn("resolution_action_updated_date", F.to_timestamp("resolution_action_updated_date"))
    print("New Data Count:", df_new_data.count())
    DeltaTable.forPath(spark, "data_515.default.311_service_requests") \
    .alias("target") \
    .merge(
        df_new_data.alias("source"),
        "target.unique_key = source.unique_key"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()
    print("Merge completed")
    
else:
    print("No new data")
